# Torch.nn 

The package ```torch.nn``` contains utilities for building the kind of large, complex neural networks we've seen in the lectures. It is built around <em>modules</em>: modules are classes that combine a set of weights with a ```forward()``` function that uses these weights to compute a forward pass.

The simplest module is probably ```torch.nn.Linear```. It implements a simple linear operation with a weight matrix and a bias vector (this is like the ```Dense``` layer in Keras).

A biblioteca ```torch.nn``` contém utilitários para construir o tipo de redes neurais grandes e complexas que vimos nas aulas. Ela é construída em torno de <em>módulos</em>: módulos são classes que combinam um conjunto de pesos com uma função ```forward()``` que utiliza esses pesos para realizar um passe adiante.

O módulo mais simples provavelmente é o ```torch.nn.Linear```. Ele implementa uma operação linear simples com uma matriz de pesos e um vetor de viés (isso é semelhante à camada ```Dense``` no Keras).

In [2]:
import torch
from torch import nn

lin = nn.Linear(2,2) # Uma função linear de um vetor 2D para um vetor 2D.

print('Peso da Matriz: ', lin.weight)
print('Bias do vetor: ', lin.bias)

Peso da Matriz:  Parameter containing:
tensor([[ 0.5804, -0.4330],
        [-0.5799,  0.5503]], requires_grad=True)
Bias do vetor:  Parameter containing:
tensor([0.5278, 0.0748], requires_grad=True)


You can see that Pytorch knows that ```lin.weight``` and ```lin.bias``` are _parameters_. This is because they are ```nn.Parameter``` objects, a lightweight wrapper around the torch tensor, which signals that this tensor is meant to be treated like a model parameter. It has ```requires_grad=True``` by default, and there are helper functions to collect all parameters of a complex model.

We can apply the linear transformation by calling ```lin``` just like a function.


Você pode ver que o Pytorch reconhece que ```lin.weight``` e ```lin.bias``` são _parâmetros_. Isso ocorre porque eles são objetos ```nn.Parameter```, uma camada leve em torno do tensor do torch, que indica que esse tensor deve ser tratado como um parâmetro do modelo. Por padrão, ele tem ```requires_grad=True```, e existem funções auxiliares para coletar todos os parâmetros de um modelo complexo.

Podemos aplicar a transformação linear chamando ```lin``` como se fosse uma função.

In [3]:
x = torch.tensor([1.0, 2.0])
lin(x)

tensor([0.2421, 0.5956], grad_fn=<AddBackward0>)

Note that the resulting tensor has a ```grad_fn``` attribute, so we can tell that the computation graph is being remembered.

To implement a module of our own, we create a subclass of the ```nn.Module``` class. All we need to implement is the constructor and the ```forward``` function. Here is a module for a simple two-layer MLP with a ReLU activation on the hidden layer.

To illustrate how to define and apply parameters, we will also add a multiplier to the output (a single learnable value). 

Observe que o tensor resultante possui um atributo ```grad_fn```, o que nos permite saber que o grafo de computação está sendo lembrado.

Para implementar nosso próprio módulo, criamos uma subclasse da classe ```nn.Module```. Tudo que precisamos implementar é o construtor e a função ```forward```. Aqui está um módulo para uma MLP (Multilayer Perceptron) simples de duas camadas com ativação ReLU na camada oculta.

Para ilustrar como definir e aplicar parâmetros, também adicionaremos um multiplicador à saída (um único valor aprendível).

In [10]:
import torch.nn.functional as F # some helpful utility functions 

# Criando a Multicamada Perceptron

class MLP(nn.Module):

    def __init__(self, in_size = 16, hidden_size = 32, out_size = 1 ) -> None:
        '''
        This is the _constructor_ the function that creates an instace of the MLP class.

        The argument 'self' is a standard argument in python object-oriented programing. It refers to the current instance that we're creating. The other arguments are parameters of the MLP

        Isso é o _construtor_, a função que cria uma instância da classe MLP.
        
        O argumento 'self' é um argumento padrão na programação orientada a objetos em Python. Ele se refere à instância atual que estamos criando. Os outros argumentos são parâmetros do MLP.
        '''
        super().__init__()

        # everything that has(possui) parameters should(deve) be created in the constructor

        self.layer1 = nn.Linear(in_size, hidden_size)
        self.layer2 = nn.Linear(hidden_size, out_size)

        # the layers have(têm) most(maioria) of the parameters, but we will also add one of our own(autoria)

        self.mult = nn.Parameter(torch.tensor([1.0]))

        # -- we create a tensor with the initial value, and wrap(envolvemos) it in an nn.Parameter objects. Because(por ser)it's an nn.Parameter, pytorch will take care of the rest(cuidará do resto)

    def forward(self, x):
        '''
        This is the function (que é executada)that gets executed when we call the module like(como) a function.

        - The argument 'self' again(novamente) refers to the current object.
        - The argument 'x' is the input to the function(multiple arguments, named(nomeados)arguments and even no arguments are(são) possible)
        '''
        h = self.layer1(x) # x no formato (_,_)
        h = F.relu(h) # apply a reLU nonlinearity 
        o = self.layer2(h) # apply second layer 
        o = o * self.mult # apply the multplier(multiplicador)

        return o


We can(podemos) now created an MLP instance, and feed(alimentá-la) it some data. Before you run the cell(célula), can(pode) you predict the size of the output? 

In [5]:
mlp = MLP() # create an MLP with the standard dimensions

x = torch.randn(3,16) # three instances, with 16 features

mlp(x) # pass the data through the MLP

O tamanho da saída será: torch.Size([3, 1])


tensor([[0.0489],
        [0.1074],
        [0.1480]], grad_fn=<MulBackward0>)

Because we've subclassed ```nn.Module```, we get a lot of functionality for free. For instance, ```mlp``` has a function that lets us loop over all its parameters and the parameters of its modules (and the parameters of their modules and so on).

Como nós criamos uma subclasse de ```nn.Module```, nós obtemos muitas funcionalidades gratuitamente. Por exemplo, ```mlp``` possui uma função que nos permite percorrer todos os seus parâmetros e os parâmetros de seus módulos (e os parâmetros dos módulos deles e assim por diante).

In [6]:
for param in mlp.parameters():
    print(param.size())

torch.Size([1])
torch.Size([32, 16])
torch.Size([32])
torch.Size([1, 32])
torch.Size([1])


In order, these are the multiplier, the weights matrix of the first layer, the bias of the first layer, the weight matrix of the second layer and the bias of the second layer.

This is helpful when we need to let the optimizer know what the parameters of our model are. 

Here's an example of how to put everything together and train our MLP on some generated data

Em ordem, estes são o multiplicador, a matriz de pesos da primeira camada, o viés da primeira camada, a matriz de pesos da segunda camada e o viés da segunda camada.

Isso é útil quando precisamos informar ao otimizador quais são os parâmetros do nosso modelo.

Aqui está um exemplo de como juntar tudo e treinar nosso MLP em alguns dados gerados.

In [16]:
import torch
from torch.optim import Adam

# hyperparameters
iterantions = 1000
learning_rate = 0.01

# regenarate model and data 

x = torch.randn(1000, 32)

# we'll use the vector norm as a target function
t = torch.sqrt(x.pow(2).sum(dim=1, keepdim = True)) # parametro keepdim(manter dimensões)

model = MLP(32,64,1)

opt = Adam(lr=learning_rate, params=model.parameters())
# -- Note that we just point the optimizer to the parameters generator 
# Observem que apenas direcionamos o otimizador para o gerador de parâmetros.

for i in range(iterantions):

    y = model(x)
    loss = F.mse_loss(y, t)

    #  Vamos mudar para a implementação do MSE usando o PyTorch.

    if i % 50 == 0:
        print(f'iteration{i: 4} loss {loss:.4}')
    
    loss.backward()
    opt.step()
    opt.zero_grad()



iteration   0 loss 32.57
iteration  50 loss 0.2538
iteration 100 loss 0.05096
iteration 150 loss 0.02284
iteration 200 loss 0.01234
iteration 250 loss 0.007344
iteration 300 loss 0.004753
iteration 350 loss 0.003202
iteration 400 loss 0.002201
iteration 450 loss 0.001569
iteration 500 loss 0.001139
iteration 550 loss 0.0008399
iteration 600 loss 0.000627
iteration 650 loss 0.0004676
iteration 700 loss 0.0003507
iteration 750 loss 0.0002648
iteration 800 loss 0.0002012
iteration 850 loss 0.0001529
iteration 900 loss 0.0001172
iteration 950 loss 8.821e-05
